In [19]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Annotated, Literal, Optional
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

True

In [13]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [23]:
system_message = SystemMessage(content="You are an assistant that answers questions or replies within 20 words or less.")

In [24]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    

In [25]:
def chat_node(state: ChatState) -> ChatState:
    # Prepend for the model only. Do not return it, so it is never stored or shown.
    messages = [system_message] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

In [26]:
checkpoint = MemorySaver()

graph = StateGraph(ChatState)
graph.add_node("chat", chat_node)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)

chatbot = graph.compile(checkpointer=checkpoint)


In [ ]:
thread_id = "1"
config = {"configurable": {"thread_id": thread_id}}

initial_state = {
    "messages": [
        HumanMessage(content="Hello, how are you?")
    ]
}

result = chatbot.invoke(initial_state, config=config)
print(result)


{'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='670557e8-5894-4448-b35c-d5d21cf1adb8'), AIMessage(content="Hello! I'm just a digital assistant, so I don't have feelings, but I'm here and ready to help you with anything you need. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 13, 'total_tokens': 50, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EVhvocq6Ufoa6TsLCh70A3G54Tx0k', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a10d5f-d10c-7592-985b-6841a6bb1ea3-0', tool_calls=[], invalid_tool_calls=[], usage_met

In [16]:
result["messages"][-1].content

"Hello! I'm just a digital assistant, so I don't have feelings, but I'm here and ready to help you with anything you need. How can I assist you today?"

In [27]:
# unique identifier for each thread/conversation
thread_id = "1"

while True:
    user_input = input("You: ")
    print("user input: ", user_input)
    if user_input.lower() in ["exit", "quit"]:
        break
    config = {"configurable": {"thread_id": thread_id}}
    result = chatbot.invoke({"messages": [HumanMessage(content=user_input),]}, config=config)
    print("bot: ", result["messages"][-1].content)


user input:  hello i m hev
bot:  Hello, Hev! How can I assist you today?
user input:  what is the capital of romania
bot:  The capital of Romania is Bucharest.
user input:  why do small robots toys are not yet commercialized(bipedal ones)
bot:  Bipedal robots face challenges in stability, cost, and technology development, hindering commercialization.
user input:  how rich are you?
bot:  I don't have personal wealth; I'm just a program designed to assist you.
user input:  what is my name?
bot:  You mentioned your name is Hev.
user input:  exit


In [28]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hello i m hev', additional_kwargs={}, response_metadata={}, id='1ab2ba0c-7c2c-4697-9ebe-796d5e6d008e'), AIMessage(content='Hello, Hev! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 31, 'total_tokens': 43, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_169c0f824d', 'id': 'chatcmpl-EVihKw2M1FhsnnEV22aua584NXFFj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a10d8c-cad8-7990-84eb-30e5e4abf0ba-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 12, 'total_tokens': 43, 'input_token